<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/deep_learning/Text_Generation_with_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Generation with Transformers (GPT-2)

## Introduction

Language models learn to predict the probability of the next word given all previous words:

$$P(w_1, w_2, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \ldots, w_{t-1})$$

**GPT-2** (Radford et al., 2019) is a decoder-only Transformer pre-trained on 40GB of internet text with this **autoregressive** objective. It can generate coherent long-form text by iteratively predicting the next token.

## The Transformer Architecture

### Self-Attention

The core building block is **scaled dot-product attention**:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- $Q, K, V \in \mathbb{R}^{T \times d_k}$: query, key, value matrices
- Dividing by $\sqrt{d_k}$ prevents the dot products from growing too large (which would push softmax into saturation)
- Each token attends to all other tokens, weighting by relevance

### Multi-Head Attention

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

Multiple heads allow the model to jointly attend to information from different representation subspaces.

### Causal (Decoder) Masking

In a language model, token $t$ may only attend to tokens $1, \ldots, t$ (not future tokens). This is enforced by a causal mask: setting attention weights to $-\infty$ for positions $j > i$.

### Positional Encoding

Transformers process all tokens in parallel (no recurrence), so they have no inherent notion of order. GPT-2 uses **learned positional embeddings** added to the token embeddings.

## GPT-2

| Model | Parameters | Layers | Heads | $d_{\text{model}}$ |
|-------|-----------|--------|-------|---------------------|
| small | 117M | 12 | 12 | 768 |
| medium | 345M | 24 | 16 | 1024 |
| large | 762M | 36 | 20 | 1280 |
| XL | 1.5B | 48 | 25 | 1600 |

At each position $t$, GPT-2 outputs a logit vector $\mathbf{z}_t \in \mathbb{R}^{|V|}$ (vocabulary size $|V| = 50{,}257$). The predicted next-token distribution is:

$$P(w_{t+1} \mid w_{1:t}) = \text{softmax}(\mathbf{z}_t)$$

## Tokenization: Byte-Pair Encoding (BPE)

GPT-2 uses **BPE tokenization** — a subword algorithm that:
1. Starts with individual characters
2. Iteratively merges the most frequent adjacent pair
3. Produces a vocabulary of common subwords

This avoids both character-level (too fine) and word-level (too large vocabulary, can't handle rare words) tokenization.

## Notation

| Symbol | Meaning |
|--------|---------|
| $T$ | Sequence length (number of tokens) |
| $|V| = 50{,}257$ | GPT-2 vocabulary size |
| $\mathbf{z}_t \in \mathbb{R}^{|V|}$ | Logit vector at position $t$ |
| $\tau$ | Temperature parameter for sampling |

## Roadmap

1. Tokenize input text with GPT-2's BPE tokenizer
2. Run a forward pass and inspect per-position predictions
3. Visualise top-k next-token probabilities
4. Generate full paragraphs with greedy and temperature sampling

# Install Dependencies

In [1]:
!pip install pytorch-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.5 MB/s eta 0:00:00


## Load GPT-2

We load the pre-trained tokenizer and language model. The model weights are downloaded from the Hugging Face hub (~500MB for GPT-2 small).

In [2]:
import torch
from pytorch_transformers import GPT2Tokenizer, GPT2LMHeadModel

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

100%|██████████| 456318/456318 [00:00<00:00, 1886584.71B/s]


In [4]:
print(tokenizer)

## Next-Token Prediction

### How GPT-2 processes text

GPT-2 takes a sequence of token indices as input. For a sequence of length $T$, it outputs a tensor of shape $(1, T, |V|)$ — logits for the next token at every position.

In [5]:
text = "Welcome to the deep learning class, it is a"
indexed_tokens = tokenizer.encode(text)
indexed_tokens

[19134, 284, 262, 2769, 4673, 1398, 11, 340, 318, 257]

In [6]:
print(tokenizer.decode(indexed_tokens))

 Welcome to the deep learning class, it is a


In [7]:
tokens_tensor = torch.tensor([indexed_tokens])
tokens_tensor

tensor([[19134,   284,   262,  2769,  4673,  1398,    11,   340,   318,   257]])

In [8]:
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

100%|██████████| 548118077/548118077 [00:17<00:00, 31183818.40B/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Move to GPU for faster inference (skip if no CUDA available):

In [9]:
tokens_tensor = tokens_tensor.to('cuda')
model.to('cuda')

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [10]:
with torch.no_grad():
    outputs = model(tokens_tensor)
    predictions = outputs[0]

In [11]:
predictions.shape

torch.Size([1, 10, 50257])

At each position $t$, `predictions[0, t, :]` is the logit vector for the $(t+1)$-th token. The model is trained to predict the next word autoregressively.

At each step of the sequence, GPT2 is trained to predict the next word.

In [12]:
for i in range(predictions.shape[1]):
    predicted_index = torch.argmax(predictions[0, i, :]).item()
    indexed_tokens_i = indexed_tokens[:i+1] + [predicted_index]
    predicted_text = tokenizer.decode(indexed_tokens_i[:-1]) + f"({tokenizer.decode(indexed_tokens_i[-1])})"
    print(f"With {i} input -> {predicted_text}")


With 0 input ->  Welcome( to)
With 1 input ->  Welcome to( the)
With 2 input ->  Welcome to the( latest)
With 3 input ->  Welcome to the deep( end)
With 4 input ->  Welcome to the deep learning( community)
With 5 input ->  Welcome to the deep learning class(.)
With 6 input ->  Welcome to the deep learning class,( where)
With 7 input ->  Welcome to the deep learning class, it('s)
With 8 input ->  Welcome to the deep learning class, it is( a)
With 9 input ->  Welcome to the deep learning class, it is a( great)


In [13]:
print(text)

Welcome to the deep learning class, it is a


The token with the highest logit at the final position is the model's best single next-word prediction (greedy decoding):

Therefore, the predicted next word would be:

In [14]:
predicted_index = torch.argmax(predictions[0, -1, :]).item()
predicted_text = tokenizer.decode(indexed_tokens + [predicted_index])
predicted_text

' Welcome to the deep learning class, it is a great'

## Top-k Token Probabilities

`predictions[0, -1, :]` is the logit vector for the next token after our full input. Applying softmax converts logits to probabilities. Let's see the top-10 candidates:

predictions[0, -1, :] is a vector that represents the probability of each of the 50257 tokens being the next word (at the end of the sequence for the -1). Let's see which are the top 10 most likely according to GPT2.

In [15]:
import torch.nn.functional as F

top_k = 10
predicted_texts = []

# Apply softmax to the predictions
softmax_predictions = F.softmax(predictions[0, -1, :], dim=-1)

for k in range(top_k):
    # Get the k-th largest index and its probability
    predicted_index = torch.topk(softmax_predictions, k + 1).indices[-1].item()
    predicted_prob = softmax_predictions[predicted_index].item()

    # Print the decoded token and its probability
    decoded_token = tokenizer.decode(indexed_tokens + [predicted_index])
    print(f"{decoded_token} (Probability: {predicted_prob:.4f})")

    # Optionally, you can store the results in a list
    predicted_texts.append(decoded_token)


 Welcome to the deep learning class, it is a great (Probability: 0.1839)
 Welcome to the deep learning class, it is a fun (Probability: 0.0433)
 Welcome to the deep learning class, it is a good (Probability: 0.0377)
 Welcome to the deep learning class, it is a very (Probability: 0.0377)
 Welcome to the deep learning class, it is a place (Probability: 0.0164)
 Welcome to the deep learning class, it is a class (Probability: 0.0158)
 Welcome to the deep learning class, it is a lot (Probability: 0.0134)
 Welcome to the deep learning class, it is a really (Probability: 0.0125)
 Welcome to the deep learning class, it is a wonderful (Probability: 0.0125)
 Welcome to the deep learning class, it is a fantastic (Probability: 0.0122)


## Autoregressive Text Generation

### Greedy Decoding

At each step, feed the current sequence to GPT-2, take the $\arg\max$ of the output logits, append that token, and repeat.

**Problem:** greedy decoding tends to produce repetitive, high-probability sequences. It never takes the "second-best" path.

Often times, in generative AI we sample from this distribution (instead of picking just the most probable one) in order to generate "creative" answers.

Finally, if we wanted to generate an entire paragraph, we would only need to generate the next word and then feedback the network back with the larger sentence.

In [16]:
start = 'Studying the MINT master is the best decision I'
indexed_tokens = tokenizer.encode(start)

for i in range(25):
  tokens_tensor = torch.tensor([indexed_tokens])
  tokens_tensor = tokens_tensor.to('cuda')
  with torch.no_grad():
    outputs = model(tokens_tensor)
    predictions = outputs[0]
    predicted_index = torch.argmax(predictions[0, -1, :]).item()
    indexed_tokens = indexed_tokens + [predicted_index]

In [17]:
predicted_text = tokenizer.decode(indexed_tokens + [predicted_index])
print(predicted_text)

 Studying the MINT master is the best decision I've ever made. I'm not sure if I'm going to be able to do it, but I'm going to be be


## Temperature Sampling

Instead of always picking the most likely token, we **sample** from the distribution. **Temperature** $\tau$ controls how "peaked" the distribution is:

$$P_\tau(w) \propto \exp\!\left(\frac{\log P(w)}{\tau}\right) = P(w)^{1/\tau}$$

- $\tau \to 0$: concentrates mass on the single most likely token (→ greedy)
- $\tau = 1$: original model distribution
- $\tau > 1$: flatter distribution → more diverse but less coherent

### Top-k Sampling

We further restrict sampling to only the $k$ most likely tokens at each step, which prevents sampling very unlikely tokens that would derail the text.

In [18]:
TEMPERATURE = 0.1

In [19]:
def generate_text(start, sentence_length, chose_between):
    indexed_tokens = tokenizer.encode(start)
    for i in range(sentence_length):
        tokens_tensor = torch.tensor([indexed_tokens])
        tokens_tensor = tokens_tensor.to('cuda')
        with torch.no_grad():
            outputs = model(tokens_tensor)
            predictions = outputs[0] / TEMPERATURE

            top_indices = torch.topk(predictions[0, -1, :], chose_between).indices
            top_probs = F.softmax(predictions[0, -1, top_indices], dim=-1)

            sampled_index = top_indices[torch.multinomial(top_probs, 1).item()]

            if sampled_index.item() is not None:
                indexed_tokens = indexed_tokens + [sampled_index.item()]

    generated_text = tokenizer.decode(indexed_tokens)
    return generated_text

In [20]:
# Generate 10 different samples
for i in range(10):
    generated_sample = generate_text('Studying the MINT master is the best decision I', 20, 3)
    print(f"Sample {i+1}:\n{generated_sample}\n")

Sample 1:
 Studying the MINT master is the best decision I've ever made. I'm not sure if I'm going to be able to do it, but

Sample 2:
 Studying the MINT master is the best decision I've ever made.

I'm not sure if I'm going to be able to do it

Sample 3:
 Studying the MINT master is the best decision I've ever made. I'm not sure if I'm going to be able to do it, but

Sample 4:
 Studying the MINT master is the best decision I've ever made.

I'm not sure if I'm going to be able to do it

Sample 5:
 Studying the MINT master is the best decision I've ever made. I'm not sure if I'm going to be able to do it, but

Sample 6:
 Studying the MINT master is the best decision I've ever made.

I'm not sure if I'm going to be able to do it

Sample 7:
 Studying the MINT master is the best decision I've ever made.

I'm not sure if I'm going to be able to do it

Sample 8:
 Studying the MINT master is the best decision I've ever made. I'm not sure if I'm going to be able to do it, but

Sample 9:
 Study

## Reflection Questions

1. **Autoregressive vs masked LM.** GPT-2 is trained as an *autoregressive* language model (predict next token). BERT uses *masked* language modeling (predict randomly masked tokens). What are the key differences and what tasks is each better suited for?

2. **Temperature effects.** Lower temperature concentrates probability mass, higher temperature spreads it. At $\tau = 0.1$ (nearly greedy), what do you notice about the generated texts? At $\tau = 2.0$ (high entropy), what might you see?

3. **Context window.** GPT-2 small has a context window of 1,024 tokens. What happens to generation quality when the prompt + generated text exceeds this limit? How do modern LLMs (e.g., GPT-4) address this?

4. **Exposure bias.** During training, the model sees *ground-truth* previous tokens. During generation, it sees its *own* predictions (which may be wrong). What is this discrepancy called, and how does it affect generation quality for longer sequences?

## Answers

**1. Autoregressive vs masked LM.**
Autoregressive models (GPT-2) are natural for *generation* tasks (text completion, story writing). Masked LMs (BERT) are better for *understanding* tasks (classification, NER, QA) because they see bidirectional context. GPT-2 can only attend to *past* tokens; BERT attends to all tokens.

**2. Temperature effects.**
At $\tau = 0.1$: nearly deterministic — the same text every run, often repetitive. At $\tau = 2.0$: highly varied but potentially incoherent, grammatically incorrect, or off-topic.

**3. Context window.**
Beyond the context window, the model cannot attend to earlier text and "forgets" the beginning. Modern LLMs extend context via sparse attention, sliding windows, or architectural innovations like RoPE positional embeddings that generalise beyond training length.

**4. Exposure bias.**
This discrepancy is called **exposure bias**. Errors in early tokens compound — a wrong word at step 5 leads to a degraded context for step 6, and so on. **Scheduled sampling** and **reinforcement learning from human feedback (RLHF)** are techniques used to mitigate this.